to setup python environment with needed packages:
  
conda create -n openai-cloud   
conda activate openai-cloud   
conda install -c conda-forge openai>=1.2.4 ipykernel jupyterlab notebook python=3.10.0   

to set up kernel in jupyterlabs:  

python -m ipykernel install --user --name=openai-cloud  

reference:  
https://help.openai.com/en/articles/5112595-best-practices-for-api-key-safety  


In [ ]:
!pip install -U openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 668.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 8.1 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from openai import OpenAI
import base64
import json
import os
from urllib.parse import urlparse

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [ ]:
import json
import os

def process_response(response, image_local):

    message = response.choices[0].message

    # Check whether the model refused the request
    if message.refusal:
        print("Request refused by the model:")
        print(message.refusal)
        return

    # Check whether content exists
    if message.content is None:
        print("No content returned by the model.")
        return

    json_string = message.content

    # Convert JSON string to Python dictionary
    json_data = json.loads(json_string)

    filename_without_extension = os.path.splitext(
        os.path.basename(image_local)
    )[0]

    json_filename = f"{filename_without_extension}.json"

    output_path = (
        "/content/drive/MyDrive/PhdWork/Upgrad/Data/"
        + json_filename
    )

    with open(output_path, "w") as file:
        json.dump(json_data, file, indent=4)

    print(f"JSON data saved to {output_path}")

In [ ]:
# image_local = 'ML-1955-2.png'
image_local = '/content/drive/MyDrive/PhdWork/Upgrad/passport_sample.jpg'
base64_img = encode_image(image_local)
#image_url = 'https://d2908q01vomqb2.cloudfront.net/f1f836cb4ea6efb2a0b1b99f41ad8b103eff4b59/2021/02/19/ML-1955-2.jpg'


In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI() #Best practice needs OPENAI_API_KEY environment variable
#client = OpenAI('OpenAI API Key here')

In [ ]:
response=client.chat.completions.create(
    model='gpt-6-luna',
    messages=[{"role":"user",
               "content": [{
                   "type":"text",
                   "text": "Return JSON document with data. Only return JSON not other text"
                   },
                     {
                      "type":"image_url",
                       "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_img}"
                       }

                     }


               ],
               }],
    response_format={"type": "json_object"},
    max_completion_tokens=500)

# 3. Process and save the response
print(response)


ChatCompletion(id='chatcmpl-EUoBCI82owqYMgsbM3VcaZ88ZZ3GR', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "document_type": "Passport",\n  "country": "India",\n  "passport_number": "SP003369",\n  "surname": "THAPLIYAL",\n  "given_names": "GARIMA",\n  "nationality": "INDIAN",\n  "date_of_birth": "01/07/1994",\n  "sex": "F",\n  "place_of_birth": "DELHI, DELHI",\n  "place_of_issue": "COIMBATORE",\n  "date_of_issue": "03/09/2024",\n  "date_of_expiry": "02/09/2034"\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1791011438, model='gpt-6-luna', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=315, prompt_tokens=147, total_tokens=462, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=172, rejected_prediction_toke

In [ ]:
process_response(response, image_local)

JSON data saved to /content/drive/MyDrive/PhdWork/Upgrad/Data/passport_sample.json
